In [ ]:

from pathlib import Path
from itertools import product
import shlex

exp_dir_str = "./experiments_1027"
exp_dir = Path(exp_dir_str)
exp_dir.mkdir(parents=True, exist_ok=True)

# 保存先
script_dir = Path(exp_dir, "scripts")
script_dir.mkdir(parents=True, exist_ok=True)
results_dir = Path(exp_dir, "results")
results_dir.mkdir(parents=True, exist_ok=True)

# 全実験で共通の引数
base_args = [
    "--data_dir", "./data_collection/Place2Vec_center/",
    "--num_context_sample", "10",
    "--embed_dim", "64",
    "--dropout", "0.5",
    "--enc_agg", "mean",
    "--model_type", "global",
    "--num_rbf_anchor_pts", "0",
    "--spa_enc", "wlmultisol",
    "--spa_embed_dim", "64",
    "--freq", "16",
    "--max_radius", "10000",
    "--min_radius", "100",
    "--spa_f_act", "sigmoid",
    "--freq_init", "geometric",
    "--spa_enc_use_layn", "F",
    "--spa_enc_use_postmat", "T",
    "--g_spa_embed_dim", "64",
    "--g_freq", "32",
    "--g_max_radius", "40000",
    "--g_min_radius", "50",
    "--g_spa_f_act", "relu",
    "--g_freq_init", "geometric",
    "--g_spa_enc_use_layn", "T",
    "--g_spa_enc_use_postmat", "T",
    "--num_hidden_layer", "1",
    "--hidden_dim", "512",
    "--use_layn", "T",
    "--skip_connection", "T",
    "--use_dec", "T",
    "--init_decoder_atten_type", "concat",
    "--init_decoder_atten_act", "leakyrelu",
    "--init_decoder_atten_f_act", "sigmoid",
    "--init_decoder_atten_num", "1",
    "--init_decoder_use_layn", "T",
    "--init_decoder_use_postmat", "T",
    "--decoder_atten_type", "concat",
    "--decoder_atten_act", "leakyrelu",
    "--decoder_atten_f_act", "sigmoid",
    "--decoder_atten_num", "1",
    "--decoder_use_layn", "T",
    "--decoder_use_postmat", "T",
    "--join_dec_type", "max",
    "--act", "sigmoid",
    "--opt", "adam",
    "--lr", "0.001",
]

# 実験条件
seeds = range(10)
# wl_funcs = [
#     "haar",
#     "db4",
#     "db8",
#     "sym4",
#     "coif2",
#     "bior4.4",
# ]
wl_funcs = [
    "db4",
    "db8"
]
min_scales = [500]
stg_sigmas = [0.5]
stg_lambdas = [1]
use_stg_options = [False, True]

# STGなしではsigma, lambdaを変化させない
configs = []
for use_stg, wl, scale, seed in product(
    use_stg_options, wl_funcs, min_scales, seeds
):
    pairs = (
        product(stg_sigmas, stg_lambdas)
        if use_stg else [(None, None)]
    )
    for sigma, lam in pairs:
        configs.append((use_stg, wl, scale, seed, sigma, lam))

for use_stg, wl, scale, seed, sigma, lam in configs:

    name = (
        f"wl_1022_{wl}_scale_{scale}_"
        f"stg_{int(use_stg)}"
    )
    if use_stg:
        name += f"_sigma_{sigma}_lambda_{lam}"
    name += f"_seed_{seed}"

    args = [
        *base_args,
        "--model_dir", f"{exp_dir_str}/results",
        "--log_dir", f"{exp_dir_str}/results",
        "--seed", str(seed),
        "--g_spa_enc", "wlmultisol",
        "--wl_func", wl,
        "--min_scale", str(scale),
    ]

    if use_stg:
        args += [
            "--stg_sigma", str(sigma),
            "--stg_lambda", str(lam),
            "--use_stg",
        ]

    args += [
        "--max_iter", "2500",
        "--batch_size", "512",
        "--log_every", "50",
        "--val_every", "50",
        "--cuda",
    ]

    lines = [
    "python -m spacegraph_codebase.Place2Vec.train"
    ]

    i = 0
    while i < len(args):
        if i + 1 < len(args) and not args[i + 1].startswith("--"):
            lines.append(
                f"  {shlex.quote(args[i])} {shlex.quote(args[i + 1])}"
            )
            i += 2
        else:
            lines.append(f"  {shlex.quote(args[i])}")
            i += 1

    command = " \\\n".join(lines)

    script = f"""#!/bin/bash
    set -e
    {command}
    """
    

    path = script_dir / f"{name}.sh"
    path.write_text(script)

print(f"Generated {len(configs)} scripts in {script_dir}")


In [2]:
from pathlib import Path
import re

exp_dir = Path("./experiments_1027")
script_dir = exp_dir / "scripts"
script_dir.mkdir(parents=True, exist_ok=True)

# 元のシェルスクリプト
base_script = Path("./experiments_1027/scripts/Place2Vec_2_enc_dec_global_theory.sh")
template = base_script.read_text()

for seed in range(1, 10):
    script = re.sub(
        r"--seed\s+\d+",
        f"--seed {seed}",
        template
    )

    output_path = script_dir / f"theory_seed_{seed}.sh"
    output_path.write_text(script)

print("Generated 9 scripts")

Generated 9 scripts
